# Session 10 Lab: Embeddings and Semantic Search
**Course:** Noob to AI Expert | **Track:** Intermediate

Build a semantic search engine over course notes using OpenAI embeddings and cosine similarity.

In [ ]:
!pip install openai numpy chromadb umap-learn matplotlib -q
print('✅ Ready')

In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()

course_notes = [
    'Prompt engineering shapes model behavior through instructions, context, examples, and constraints.',
    'Python lists store ordered collections; dicts store key-value pairs.',
    'Machine learning finds patterns in data by minimizing a loss function using gradient descent.',
    'Overfitting occurs when a model memorizes training data instead of learning generalizable patterns.',
    'Precision measures correct positive predictions; recall measures how many positives were found.',
    'Neural networks are composed of layers of neurons with weights adjusted via backpropagation.',
    'Transformers use self-attention to capture relationships between all tokens simultaneously.',
    'Hallucination happens when an LLM generates plausible-sounding but factually incorrect text.',
    'Cosine similarity measures the angle between two embedding vectors as a proxy for semantic similarity.',
    'RAG retrieves relevant document chunks and passes them as context to the LLM before generation.',
]

def embed_texts(texts):
    resp = client.embeddings.create(model='text-embedding-3-small', input=texts)
    return [d.embedding for d in resp.data]

doc_embeddings = embed_texts(course_notes)
print(f'Embedded {len(doc_embeddings)} notes, each {len(doc_embeddings[0])} dimensions')

In [ ]:
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query, top_k=3):
    q_emb = embed_texts([query])[0]
    scores = [(course_notes[i], cosine_sim(q_emb, e)) for i, e in enumerate(doc_embeddings)]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

for query in ['how do models learn from mistakes', 'what causes wrong AI outputs', 'similarity between concepts']:
    print(f'Query: "{query}"')
    for note, score in semantic_search(query):
        print(f'  [{score:.3f}] {note[:80]}')
    print()

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(np.array(doc_embeddings))

plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=100, c=range(len(course_notes)), cmap='plasma')
for i, note in enumerate(course_notes):
    plt.annotate(note[:40]+'...', (coords[i, 0], coords[i, 1]), fontsize=7, ha='center', va='bottom')
plt.title('Embedding Space Visualization (UMAP)')
plt.axis('off')
plt.show()